# 9 — D₀, the model against the data, and the baselines

The analysis half of the model work. Everything here reads the artifacts that
`08_simulations.ipynb` produced and runs in seconds; nothing here runs the
simulator.

Produces **Figure 4**, **Figure S5**, **Table S7** and **Table S8**.

If you have the simulation artifacts already — from a previous run, or because
you were given them — this notebook works on its own and notebook 8 can be
skipped entirely.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

In [ ]:
import glob

# Both sweeps are needed and they are not interchangeable: `crossover` is the
# wide grid that records rstar_sim (figure_SI5.py, model_vs_data.py), `refined`
# is the six-seed one (estimate_d0.py). Checking only one is how a missing
# sweep used to surface three cells later as a FileNotFoundError.
sim_runs = sorted(os.path.basename(p).replace("_meta.json", "") for p in
                  glob.glob(os.path.join(REPO, "data_reduced", "sim_*_meta.json")))
sweeps = {tag: os.path.join(REPO, "outputs", f"d0_calibration_{tag}", "runs.csv")
          for tag in ("crossover", "refined")}

print("simulation runs :", sim_runs or "NONE -- run notebook 8 first")
for tag, path in sweeps.items():
    print(f"D0 sweep {tag:<10}:",
          "present" if os.path.exists(path) else "MISSING -- run notebook 8 first")

missing = [t for t, p in sweeps.items() if not os.path.exists(p)]
if not sim_runs or missing:
    raise RuntimeError(f"notebook 8 has not produced everything this one reads "
                       f"(missing sweeps: {missing or 'none'}); run it first")

## 9.1 D₀ read off the data, with no model at all

Before any calibration: D₀ is a vocabulary size, so on a rank-frequency curve it
is a position in the type ranking. Fitting a continuous two-slope power law with
a **free breakpoint** returns R*, which is directly comparable with D₀ — same
units, no conversion, no free normalisation.

This is the primary estimate in the paper, and it is model-independent.

In [ ]:
py("estimate_d0.py")

## 9.2 Table S7 — the model against the data, band for band

Four quantities, both sides at T = 10^8, reduced and fitted identically. The
model reproduces the two-regime shape and the scale where the regimes meet, and
D₀ agrees to 0.2% with the R* measured from the data alone.

The Heaps exponent is the one quantity whose bands do **not** overlap, and the
paper states that rather than hiding it.

In [ ]:
import model_vs_data

meta, dat_freq, dat_heaps, dat_wcn, sim_freq, sim_heaps, sim_wcn = model_vs_data.load()
tableS7 = model_vs_data.compare(meta, dat_freq, dat_heaps, sim_freq, sim_heaps)
cal = model_vs_data.calibration(tableS7, meta)
print(model_vs_data.write_table(tableS7, meta, cal["native"]))

## 9.3 Table S8 — the stationary baselines, met at their best

Three UMT settings, not one, because testing a baseline only where it is weakest
is not a test:

* ρ = ν gives the head exponent and no tail;
* ρ = 2 gives the tail exponent everywhere, head included;
* ρ = 2 with a large initial urn produces an apparent two-regime curve out of a
  **transient**, and is the strongest case the baseline has.

The third row matches the Heaps exponent to three decimals and the crossover by
construction, and still returns a head exponent of 0.31 against the measured
1.07. A transient is a passage between regimes, not a regime.

One caveat the table itself records: the ρ = 2, n₀ = 1 row starts from a single
item and is the only row whose vocabulary does not self-average — over four seeds
its D(T) runs from 7,141 to 29,847. Do not quote that D(T) as a property of the
model.

In [ ]:
py("stationary_baseline.py")

## 9.4 Where the model does not fit

Reported because it is a real limitation. On the learner corpus the two criteria
that estimate D₀ disagree by a factor of five, and no D₀ reproduces the learner
curve: the model does reach the empirical crossover at D₀ ≈ 300, but there both
loss terms are worse, and across the whole scan its tail exponent never
approaches the empirical value.

The likely reason is that the learner corpus is an aggregate of 3,326 texts
averaging 176 tokens each, written by many different people — a mixture of
micro-samples, whose vocabulary grows faster and whose tail is milder than any
single-stream process can be.

In [ ]:
py("model_adequacy.py")

## 9.5 Figure S5 — the two criteria that estimate D₀

One panel per criterion, both populations in each. Splitting by criterion rather
than by population is what makes the figure readable: each panel then carries one
quantity in one unit, and the comparison that matters — the two populations
behaving differently — happens inside a panel instead of across the pair.

(A) the joint Heaps-and-Zipf loss, drawn as a rise above its own minimum so that
two populations with different loss levels can share an axis; (B) the crossover
the simulation itself produces at each D₀, with a dotted horizontal at each
measured R*.

For the natives the two criteria agree to within a tenth of a decade. For the
learners they do not, and the crossover is the one to trust.

In [ ]:
py("figure_SI5.py")

## 9.6 Figure 4

The main-text summary: the calibrated model against the corpus, the two networks,
and the crossover separating two populations of speakers. Some panels also appear
in the appendix figures, which are meant to be readable on their own.

In [ ]:
import figure_4

fig4_result = figure_4.main()

## Done

Every figure and table of the paper has now been produced from `outputs/`.
`README.md` carries the map from each output file to its number in the
manuscript.